# Agricultural Data Cleaning: USDA NASS Agriculture Data

In [1]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import folium
from shapely.geometry import Point
from pathlib import Path

In [2]:
data_path = r'../../../../data/tabular/agricultural/raw'
df_ag = pd.read_csv(f'{data_path}/usdaNass-agriculture.csv')

In [3]:
# Step 1: Inspect
print(df_ag.shape)
df_ag.head()

(15456, 21)


,Program,Year,Period,Week Ending,Geo Level,State,State ANSI,Ag District,Ag District Code,County,...,Zip Code,Region,watershed_code,Watershed,Commodity,Data Item,Domain,Domain Category,Value,CV (%)
0,CENSUS,2024,END OF DEC,NaN,STATE,IOWA,19,NaN,NaN,NaN,...,NaN,NaN,0,NaN,AG LAND,"AG LAND, INCL BUILDINGS, (FOR HORTICULTURE) - ...",ORGANIZATION,"ORGANIZATION: (TAX PURPOSES, CORPORATION)","106,050,030",33.6
1,CENSUS,2024,END OF DEC,NaN,STATE,IOWA,19,NaN,NaN,NaN,...,NaN,NaN,0,NaN,AG LAND,"AG LAND, INCL BUILDINGS, (FOR HORTICULTURE) - ...",ORGANIZATION,"ORGANIZATION: (TAX PURPOSES, FAMILY & INDIVIDUAL)","168,726,456",44.7
2,CENSUS,2024,END OF DEC,NaN,STATE,IOWA,19,NaN,NaN,NaN,...,NaN,NaN,0,NaN,AG LAND,"AG LAND, INCL BUILDINGS, (FOR HORTICULTURE) - ...",ORGANIZATION,"ORGANIZATION: (TAX PURPOSES, INSTITUTIONAL & R...","18,222,600",96.9
3,CENSUS,2024,END OF DEC,NaN,STATE,IOWA,19,NaN,NaN,NaN,...,NaN,NaN,0,NaN,AG LAND,"AG LAND, INCL BUILDINGS, (FOR HORTICULTURE) - ...",ORGANIZATION,"ORGANIZATION: (TAX PURPOSES, PARTNERSHIP)","5,521,000",68.5
4,CENSUS,2024,END OF DEC,NaN,STATE,IOWA,19,NaN,NaN,NaN,...,NaN,NaN,0,NaN,AG LAND,"AG LAND, INCL BUILDINGS, (FOR HORTICULTURE) - ...",TOTAL,NOT SPECIFIED,"298,520,086",23.8


In [4]:
# Step 2: Rename columns - lowercase, replace spaces with underscores
df_ag.columns = df_ag.columns.str.strip().str.lower().str.replace(" ", "_")

# Step 3: Drop columns with >90% missing values
df_ag.replace(r'^\s*$', pd.NA, regex=True, inplace=True)
threshold = len(df_ag) * 0.10
df_ag.dropna(axis=1, thresh=threshold, inplace=True)

print("Remaining columns:", df_ag.columns.tolist())

Remaining columns: ['program', 'year', 'period', 'week_ending', 'geo_level', 'state', 'state_ansi', 'watershed_code', 'commodity', 'data_item', 'domain', 'domain_category', 'value', 'cv_(%)']


In [ ]:
# Step 4: Fix Value column
# USDA suppresses some values with special codes - replace with NaN
suppressed = {'(D)', '(H)', '(Z)', '(NA)', ' (D)', ' (H)', ' (Z)', ' (NA)'}
df_ag['value'] = df_ag['value'].apply(lambda x: np.nan if str(x).strip() in suppressed else x)
df_ag['value'] = df_ag['value'].astype(str).str.replace(',', '', regex=False)
df_ag['value'] = pd.to_numeric(df_ag['value'], errors='coerce')

# Fix CV (%) column
df_ag['cv_(%)'] = df_ag['cv_(%)'].apply(lambda x: np.nan if str(x).strip() in suppressed else x)
df_ag['cv_(%)'] = pd.to_numeric(df_ag['cv_(%)'], errors='coerce')

print(f"Missing in 'value': {df_ag['value'].isnull().sum()}")
df_ag.dtypes

Missing in 'value': 3287


program                str
year                 int64
period                 str
week_ending            str
geo_level              str
state                  str
state_ansi           int64
watershed_code       int64
commodity              str
data_item              str
domain                 str
domain_category        str
value              float64
cv_(%)             float64
dtype: object

In [6]:
print(f"Shape: {df_ag.shape}")
print(f"Columns: {df_ag.columns.tolist()}")

Shape: (15456, 14)
Columns: ['program', 'year', 'period', 'week_ending', 'geo_level', 'state', 'state_ansi', 'watershed_code', 'commodity', 'data_item', 'domain', 'domain_category', 'value', 'cv_(%)']


In [7]:
df_ag.head()

,program,year,period,week_ending,geo_level,state,state_ansi,watershed_code,commodity,data_item,domain,domain_category,value,cv_(%)
0,CENSUS,2024,END OF DEC,NaN,STATE,IOWA,19,0,AG LAND,"AG LAND, INCL BUILDINGS, (FOR HORTICULTURE) - ...",ORGANIZATION,"ORGANIZATION: (TAX PURPOSES, CORPORATION)",106050030.0,33.6
1,CENSUS,2024,END OF DEC,NaN,STATE,IOWA,19,0,AG LAND,"AG LAND, INCL BUILDINGS, (FOR HORTICULTURE) - ...",ORGANIZATION,"ORGANIZATION: (TAX PURPOSES, FAMILY & INDIVIDUAL)",168726456.0,44.7
2,CENSUS,2024,END OF DEC,NaN,STATE,IOWA,19,0,AG LAND,"AG LAND, INCL BUILDINGS, (FOR HORTICULTURE) - ...",ORGANIZATION,"ORGANIZATION: (TAX PURPOSES, INSTITUTIONAL & R...",18222600.0,96.9
3,CENSUS,2024,END OF DEC,NaN,STATE,IOWA,19,0,AG LAND,"AG LAND, INCL BUILDINGS, (FOR HORTICULTURE) - ...",ORGANIZATION,"ORGANIZATION: (TAX PURPOSES, PARTNERSHIP)",5521000.0,68.5
4,CENSUS,2024,END OF DEC,NaN,STATE,IOWA,19,0,AG LAND,"AG LAND, INCL BUILDINGS, (FOR HORTICULTURE) - ...",TOTAL,NOT SPECIFIED,298520086.0,23.8


In [8]:
# Step 5: Save Cleaned Data
import os
out_path = '../../../../data/tabular/agricultural/clean'
os.makedirs(out_path, exist_ok=True)
df_ag.to_csv(f'{out_path}/usdaNass-agriculture-clean.csv', index=False)
print(f"Saved {len(df_ag):,} rows → {out_path}/usdaNass-agriculture-clean.csv")

Saved 15,456 rows → ../../../../data/tabular/agricultural/clean/usdaNass-agriculture-clean.csv
